In [1]:
import pandas as pd
import numpy as np

data = pd.read_csv("stock_prices_clean.csv", index_col=0, parse_dates=True)
returns = np.log(data / data.shift(1)).dropna()

In [2]:
# ==========================================
# Step 1: Setup for Simulation
# ==========================================

num_simulations = 10000
num_days = 252  # 1 trading year

mean_returns = returns.mean()
cov_matrix = returns.cov()

num_assets = len(mean_returns)

print("Step 1 Completed")
print("Assets:", num_assets)
print("Simulations:", num_simulations)

Step 1 Completed
Assets: 5
Simulations: 10000


In [3]:
# ==========================================
# Step 2: Monte Carlo Simulation
# ==========================================

results = np.zeros((num_simulations, 2))

for i in range(num_simulations):

    # Random weights
    weights = np.random.random(num_assets)
    weights /= np.sum(weights)

    # Portfolio return
    portfolio_return = np.sum(mean_returns * weights) * num_days

    # Portfolio volatility
    portfolio_std = np.sqrt(
        np.dot(weights.T, np.dot(cov_matrix, weights))
    ) * np.sqrt(num_days)

    results[i] = [portfolio_return, portfolio_std]

sim_df = pd.DataFrame(results, columns=['Returns', 'Volatility'])

print("Step 2 Completed")
print("Simulation Shape:", sim_df.shape)
sim_df.head()

Step 2 Completed
Simulation Shape: (10000, 2)


,Returns,Volatility
0,0.225125,0.336851
1,0.235885,0.323251
2,0.205630,0.308721
3,0.308021,0.364368
4,0.260364,0.336485


In [4]:
# ==========================================
# Step 3: Optimal Portfolio (Max Sharpe)
# ==========================================

risk_free_rate = 0.01

sim_df['Sharpe'] = (sim_df['Returns'] - risk_free_rate) / sim_df['Volatility']

best_portfolio = sim_df.loc[sim_df['Sharpe'].idxmax()]

print("Step 3 Completed")
print("Best Portfolio:")
print(best_portfolio)

Step 3 Completed
Best Portfolio:
Returns       0.354686
Volatility    0.402816
Sharpe        0.855691
Name: 3591, dtype: float64


In [5]:
# ==========================================
# Step 4: Value at Risk (VaR)
# ==========================================

# Equal weights for base portfolio
weights = np.array([1 / num_assets] * num_assets)

portfolio_returns = returns.dot(weights)

VaR_95 = np.percentile(portfolio_returns, 5)

print("Step 4 Completed")
print("Value at Risk (95%):", VaR_95)

Step 4 Completed
Value at Risk (95%): -0.034865983883059885


In [6]:
# ==========================================
# Step 5: Distribution Summary
# ==========================================

print("Simulation Summary")
print(sim_df.describe())

Simulation Summary
            Returns    Volatility        Sharpe
count  10000.000000  10000.000000  10000.000000
mean       0.261718      0.343304      0.729267
std        0.041883      0.030190      0.070068
min        0.147198      0.301048      0.413434
25%        0.230044      0.319383      0.685989
50%        0.262272      0.337981      0.741263
75%        0.290255      0.360497      0.782593
max        0.464269      0.574378      0.855691


In [7]:
# ==========================================
# Step 6: Export Data
# ==========================================

sim_df.to_csv("monte_carlo.csv", index=False)

pd.DataFrame({
    "VaR_95": [VaR_95]
}).to_csv("var.csv", index=False)

print("Phase 3 Completed Successfully")

Phase 3 Completed Successfully
